# ISU-GeoBot — results, read live from the database

Every number in Chapter 4 and on every slide, printed from the rows it was
computed from.

**Read-only where it can be.** Sections 1–6 are all `SELECT`. Nothing there can
alter the evidence it displays, and nothing is recomputed — if a cell printed
something different from the paper, the paper would be wrong. Section 7 is the
exception and says so: it runs the test suite, because a privacy boundary is
proven by executing it, not by querying it.

| Slide | Section |
|---|---|
| — | **1** · the run itself: models, parameters, row counts |
| 8 · The capability | **2** · 0 of 6 → 5 of 6, and the six questions verbatim |
| 9 · The cost in response time | **3** · per-category timings, and the overall range |
| 10 · 11 · The cost in answer quality | **4** · the four RAGAS metrics, paired |
| 6 · One answer, two origins | **5** · the classifier, its split and its features |
| 13 · Status masking · Egress boundary | **6** · the closed three-value list |
| 16 · 17 · 18 · the checks | **7** · the suite, run here |

Run the cells in order. Each answers one question a panel is likely to ask.


In [ ]:
import json, sys
from pathlib import Path
import pandas as pd

sys.path.insert(0, str(Path.cwd() if (Path.cwd() / "database_connector.py").exists()
                       else Path.cwd() / "machine-learning"))
import database_connector as db

RUN   = "b0011d70-68b1-4f81-8d80-ef3f9e0d4acc"   # run-03-simulation
MODEL = "rf-20260909-072845"                     # the served classifier

pd.set_option("display.max_colwidth", 90)

def q(sql, params=()):
    return pd.DataFrame(db.fetch_all(sql, params))

print("connected \u2713")

## 1 · Where these numbers come from

> *"How do we know this is a real evaluation and not a demo?"*

The run is stored. Note the **judge model is not the generator** — the answers were
graded by a different model from the one that wrote them.

In [ ]:
run = q("""select run_label, started_at, prompt_template_version,
              groq_model_id, llm_temperature, embedding_model, top_k,
              similarity_floor, judge_model, status_as_context
       from geobot.eval_run where id = %s""", (RUN,))
display(run.T.rename(columns={0: "value"}))

counts = q("""select
     (select count(*) from geobot.eval_query)                        as registered_queries,
     (select count(*) from geobot.eval_result where run_id = %s)     as stored_results,
     (select count(*) from geobot.ragas_score g
        join geobot.eval_result r on r.id = g.eval_result_id
        where r.run_id = %s)                                          as scored_rows""",
   (RUN, RUN))
display(counts)

### The 39 registered questions

> *"Where did the questions come from?"*

All of them, with the ground-truth answer written for each **before** any run.
The `registered_at` timestamp is what makes pre-registration checkable rather
than claimed — and `evaluation-runner.js` refuses to start a run when this
table is empty (audit C10).

In [ ]:
queries = q("""select category, query_text, ground_truth_answer, registered_at
       from geobot.eval_query order by category, registered_at""")

print(f"{len(queries)} registered questions")
for cat, grp in queries.groupby("category"):
    print(f"   {cat:24} {len(grp)}")

written = queries["ground_truth_answer"].fillna("").str.strip().ne("").sum()
print(f"\n{written} of {len(queries)} carry a written ground_truth_answer.")
print("Those were composed from the ingested documents and the point-of-interest")
print("records, never from the system's own output (thesis 3.9.1).")

shown = queries.assign(
    ground_truth=queries["ground_truth_answer"].fillna("").str.slice(0, 60) + "...",
    registered=queries["registered_at"],
)[["category", "query_text", "ground_truth", "registered"]]
display(shown.reset_index(drop=True))

## 2 · SO2 (a) — the capability finding

> *"Which queries can the Enhanced architecture answer that retrieval alone cannot?"*

This is the headline: **0 of 6 → 5 of 6**. Not a quality difference — retrieval
reaches the document corpus, and a timetable is not a document.

In [ ]:
cap = q("""select r.mode,
          count(*) as asked,
          count(*) filter (where r.answer not ilike '%%sorry%%') as answered
       from geobot.eval_result r
       join geobot.eval_query q on q.id = r.eval_query_id
       where r.run_id = %s and q.category = 'faculty_availability'
       group by r.mode order by r.mode""", (RUN,))
display(cap)

### The six questions, side by side

The last row is the one to point at. **Both arms refuse it** — but the Enhanced
arm's classifier time is `0.0 ms`, so the estimate was *never computed*, not
computed and then hidden.

In [ ]:
six = q("""select q.query_text as question, r.mode, r.answer,
              r.t_rf_ms as classifier_ms
       from geobot.eval_result r
       join geobot.eval_query q on q.id = r.eval_query_id
       where r.run_id = %s and q.category = 'faculty_availability'
       order by q.query_text, r.mode""", (RUN,))

wide = six.pivot(index="question", columns="mode", values="answer")
wide["classifier_ms (enhanced)"] = (
    six[six["mode"] == "enhanced"].set_index("question")["classifier_ms"])
display(wide)

## 3 · SO2 (b) — the cost in response time

> *"What does the enhancement cost?"*

**Read this by category, never pooled.** The navigation row is the important one:
the classifier does *zero* work there, yet the measurement still shows a gap.
That gap is the instrument's noise floor — so any end-to-end difference of that
size carries no information about the architecture.

Two columns need naming, because neither is self-explanatory:

| Column | Source | What it times |
|---|---|---|
| `classifier_ms` | `t_rf_ms` | the Random Forest call alone — the round trip to the Python service and the trees voting |
| `gates_ms` | `t_guard_ms` | the **override checks** that run *before* the model is allowed to speak: has a guard confirmed this person left, and is there an official event on record that outranks a prediction |

`gates_ms` is **not** the consent check. Consent is filtered upstream in
`intent-query-router.js` (`.eq('is_consented', true)`), so it lands in
`t_route_ms`. Both are in `faculty-presence-service.js` — `tGuard` closes at
line 541, `tRf` is measured around the `ml.predict()` call.


In [ ]:
t = q("""select q.category, r.mode,
          round(avg(r.t_total_ms)::numeric, 1) as total_ms,
          round(avg(r.t_rf_ms)::numeric, 1)    as classifier_ms,
          round(avg(r.t_guard_ms)::numeric, 1) as override_gates_ms
       from geobot.eval_result r
       join geobot.eval_query q on q.id = r.eval_query_id
       where r.run_id = %s group by q.category, r.mode""", (RUN,))

tbl = t.pivot(index="category", columns="mode", values="total_ms")
tbl["difference"] = tbl["enhanced"] - tbl["standard"]
tbl["classifier_ms"] = t[t["mode"] == "enhanced"].set_index("category")["classifier_ms"]
tbl["gates_ms"] = t[t["mode"] == "enhanced"].set_index("category")["override_gates_ms"]
display(tbl)

**The slide also says "about 1.4 seconds, between 0.8 and 2.5".** That is this
cell, not the one above — the table above is averaged *per category*, so it
cannot show the overall spread. Slowest is the number to know: a panelist who
asks "how slow does it ever get?" is asking for the maximum, not the mean.

In [ ]:
spread = q("""select r.mode,
          round(avg(r.t_total_ms)::numeric, 1) as mean_ms,
          round(min(r.t_total_ms)::numeric, 1) as fastest_ms,
          round((percentile_cont(0.5)
                 within group (order by r.t_total_ms))::numeric, 1) as median_ms,
          round((percentile_cont(0.95)
                 within group (order by r.t_total_ms))::numeric, 1) as p95_ms,
          round(max(r.t_total_ms)::numeric, 1) as slowest_ms
       from geobot.eval_result r
       where r.run_id = %s
       group by r.mode order by r.mode""", (RUN,))
display(spread.set_index("mode"))

print("1,000 ms is one second. Every figure on the response-time slide is a")
print("millisecond average of these same rows, grouped by question category.")

## 4 · SO2 (c) — the cost in RAGAS

> *"How did you compute these, and who graded them?"*

Four metrics, judged by `gpt-oss-20b` — **deliberately not the generator**. The
masked availability status is passed to RAGAS as a context item for the enhanced
arm, because it *is* retrieved context: retrieved from the classifier rather than
from pgvector. That decision is recorded on the run itself (`status_as_context`),
so it cannot be quietly changed after the fact.

**Paired means.** A query counts only where *both* arms produced a score, so the
two columns describe the same set of questions.

In [ ]:
def paired(metric):
    pair = f"""join (select r2.eval_query_id qid
        from geobot.ragas_score g2
        join geobot.eval_result r2 on r2.id = g2.eval_result_id
        where r2.run_id = %s and g2.{metric} is not null
        group by r2.eval_query_id having count(distinct r2.mode) = 2) p
        on p.qid = r.eval_query_id"""
    d = q(f"""select q.category, r.mode, round(avg(g.{metric})::numeric, 4) v, count(*) n
         from geobot.ragas_score g
         join geobot.eval_result r on r.id = g.eval_result_id
         join geobot.eval_query q on q.id = r.eval_query_id {pair}
         where r.run_id = %s and g.{metric} is not null
         group by q.category, r.mode""", (RUN, RUN))
    out = d.pivot(index="category", columns="mode", values="v").astype(float)
    out["n"] = d[d["mode"] == "enhanced"].set_index("category")["n"]
    return out

for m in ("faithfulness", "context_recall", "context_precision", "answer_relevancy"):
    print(m.replace("_", " ").title())
    display(paired(m))

**Say this before they spot it.** Context Precision is flat — it is a *retriever*
metric, and both arms share a retriever. That was predicted before it was measured.
Four metrics all rising would have been the suspicious result.

Standard scores exactly `0.0000` on availability because it produced no answer to grade.

## 5 · SO1 — the classifier

> *"What does the model look at, and how good is it?"*

**Say "simulation cohort" out loud whenever these numbers appear.**

In [ ]:
m = db.fetch_all("""select metrics, training_row_count, feature_list, algorithm,
                        class_order, split_strategy
                 from geobot.rf_model_version where version = %s""", (MODEL,))[0]
met = m["metrics"] if isinstance(m["metrics"], dict) else json.loads(m["metrics"])
fl  = m["feature_list"] if isinstance(m["feature_list"], list) else json.loads(m["feature_list"])

display(pd.DataFrame([{
    "model": MODEL,
    "algorithm": m["algorithm"],
    "accuracy": f"{met['accuracy'] * 100:.2f}%",
    "macro F1": f"{met['f1_macro']:.4f}",
    "cross-validation": f"{met['cv_f1_macro_mean']:.4f} +/- {met['cv_f1_macro_std']:.4f}",
    "train rows": f"{int(m['training_row_count']):,}",
    "test rows": f"{int(met['test_rows']):,}",
    "split": m["split_strategy"],
}]).T.rename(columns={0: "value"}))

display(pd.DataFrame(met["per_class"]).T[["precision", "recall", "f1", "support"]])

### The 11 features

Eight from the schedule, three from behaviour. The split is the argument: with
schedule features **and** schedule-derived labels, the forest would simply
reproduce the timetable lookup — it would *be* the rule baseline with a faculty
column. The three attendance features are what make it a different model.

In [ ]:
SCHEDULE = fl[:8]
ATTENDANCE = fl[8:]
display(pd.DataFrame({
    "schedule (8)":   SCHEDULE,
    "attendance (3)": ATTENDANCE + [""] * (len(SCHEDULE) - len(ATTENDANCE)),
}))

imp = db.fetch_all("select feature_importance from geobot.rf_model_version where version=%s",
                   (MODEL,))[0]["feature_importance"]
imp = imp if isinstance(imp, dict) else json.loads(imp)
s = pd.Series(imp).sort_values(ascending=False)
display(s.to_frame("mean Gini decrease").head(6))
print("\nTop feature is schedule-derived. Second is behavioural \u2014"
      " a timetable has no access to it.")

### Show me the actual trees

> *"Where is the Random Forest? Can I see a tree?"*

Everything above reads the database. This reads **the trained forest itself** —
`saved-models/rf_current.joblib`, the file the Flask service loads and serves.

In [ ]:
import joblib
from sklearn.tree import export_text

# The artifact is ~135 MB, so this cell takes a few seconds. Everything above
# read the database; this reads the trained forest itself.
bundle = joblib.load(Path(MODELS := "saved-models") / "rf_current.joblib")
forest, feature_list = bundle["model"], list(bundle["feature_list"])

print(f"{type(forest).__name__}")
print(f"  trees         {len(forest.estimators_)}")
print(f"  criterion     {forest.criterion}")
print(f"  class_weight  {forest.class_weight}")
print(f"  classes       {list(forest.classes_)}")
print(f"  trained       {bundle['trained_at']}  ({bundle['split_strategy']} split)")
print(f"  sklearn       {bundle['sklearn_version']}")

t = forest.estimators_[0]
print(f"\nTree 0 of {len(forest.estimators_)}: depth {t.get_depth()}, "
      f"{t.get_n_leaves():,} leaves, {t.tree_.node_count:,} nodes.")
print("The first three levels of its actual decision rules:\n")
print(export_text(t, feature_names=feature_list, max_depth=3, decimals=2,
                  show_weights=False))
print("Two of the first three splits are attendance features, not schedule")
print("features. A timetable lookup has no access to those — which is the")
print("whole argument for using a model here at all.")

## 6 · SO3 — what is allowed to cross the boundary

> *"You say only one coarse word reaches the language model. Show me."*

The allowlist is not a convention in the code — it is a table. Three rows, and
the middleware refuses anything that is not one of them. This cell prints the
table and then checks that the backend's own constant still agrees with it,
because a boundary that drifts from its own definition is not a boundary.

In [ ]:
states = q("""select code, display_label, thesis_label
       from geobot.availability_status order by sort_order""")
display(states)

# The code-side allowlist, read out of the middleware rather than retyped here.
# If these two ever disagree, the boundary has drifted from its own definition.
import re
mw = (Path.cwd().parent if Path.cwd().name == "machine-learning" else Path.cwd())
mw = mw / "backend" / "src" / "middleware" / "privacy-masking-middleware.js"
block = re.search(r"ALLOWED_STATUS_CODES\s*=\s*Object\.freeze\(\[(.*?)\]\)",
                  mw.read_text(encoding="utf-8"), re.S).group(1)
in_code = re.findall(r"'([a-z_]+)'", block)
in_db = list(states["code"])

print()
print("database  :", in_db)
print("middleware:", in_code)
print()
print("identical ✓" if in_db == in_code else "!! DRIFT — they disagree")
print()
print("Three values. Nothing else crosses into the language model —")
print("no probability, no confidence score, no feature vector, no room.")

## 7 · SO4 — the boundary holds

> *"How do you know it actually works?"*

**This is the one section that is not a query, and that is the point.** A
prohibition cannot be sampled — a 95% privacy boundary is a broken privacy
boundary — so it is checked by running every path and asserting the outcome.

This runs the same suite that is shown on the slides. It takes about four
seconds. The live-stack tests skip unless the backend and the ML service are
both up; that is expected and is not a failure.

In [ ]:
import os, re, subprocess

BACKEND = (Path.cwd().parent if Path.cwd().name == "machine-learning"
           else Path.cwd()) / "backend"


def run_tests(target=None):
    """The node --test summary counters.

    encoding is explicit because node writes UTF-8 and a Windows console would
    otherwise decode the summary marker as three characters of mojibake.
    """
    cmd = "npm run test:quiet" if target is None else f"node --test {target}"
    p = subprocess.run(cmd, cwd=BACKEND, shell=True, capture_output=True,
                       text=True, encoding="utf-8", errors="replace",
                       env={**os.environ, "LOG_LEVEL": "silent"})
    out = p.stdout + p.stderr
    return {k: int(v) for k, v in
            re.findall(r"(?:^|\s)(tests|suites|pass|fail)\s+(\d+)\s*$", out, re.M)}


whole = run_tests()
privacy = run_tests("tests/privacy-masking-security.test.js")

display(pd.DataFrame([
    {"suite": "the whole backend", **whole},
    {"suite": "privacy boundary only", **privacy},
]).set_index("suite"))

print()
print(f"{whole.get('pass', 0)} checks pass, {whole.get('fail', 1)} fail.")
print(f"{privacy.get('tests', 0)} of them are the masking boundary — that is")
print("tests/privacy-masking-security.test.js, one file anyone can open.")
print()
print("Slides 16 and 17 are screenshots of this exact command.")